In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df1 = pd.read_csv("Building_Permits.csv",low_memory=False)

In [5]:
df = df1[[
    "Permit Number",
    "Permit Type Definition",
    "Filed Date",
    "Issued Date",
    "Completed Date",
    "Current Status",
    "Neighborhoods - Analysis Boundaries",
    "Estimated Cost"
]]

In [6]:
df

,Permit Number,Permit Type Definition,Filed Date,Issued Date,Completed Date,Current Status,Neighborhoods - Analysis Boundaries,Estimated Cost
0,201505065519,sign - erect,05/06/2015,11/09/2015,NaN,expired,Tenderloin,4000.0
1,201604195146,sign - erect,04/19/2016,08/03/2017,NaN,issued,Tenderloin,1.0
2,201605278609,additions alterations or repairs,05/27/2016,NaN,NaN,withdrawn,Russian Hill,20000.0
3,201611072166,otc alterations permit,11/07/2016,07/18/2017,07/24/2017,complete,Nob Hill,2000.0
4,201611283529,demolitions,11/28/2016,12/01/2017,NaN,issued,Tenderloin,100000.0
...,...,...,...,...,...,...,...,...
198895,M862628,otc alterations permit,12/05/2017,12/05/2017,NaN,issued,NaN,NaN
198896,201712055595,otc alterations permit,12/05/2017,12/06/2017,NaN,issued,NaN,5000.0
198897,M863507,otc alterations permit,12/06/2017,12/06/2017,NaN,issued,NaN,NaN
198898,M863747,otc alterations permit,12/06/2017,12/06/2017,NaN,issued,NaN,NaN


In [7]:
# Understanding the Dataset

In [8]:
df.head()

,Permit Number,Permit Type Definition,Filed Date,Issued Date,Completed Date,Current Status,Neighborhoods - Analysis Boundaries,Estimated Cost
0,201505065519,sign - erect,05/06/2015,11/09/2015,NaN,expired,Tenderloin,4000.0
1,201604195146,sign - erect,04/19/2016,08/03/2017,NaN,issued,Tenderloin,1.0
2,201605278609,additions alterations or repairs,05/27/2016,NaN,NaN,withdrawn,Russian Hill,20000.0
3,201611072166,otc alterations permit,11/07/2016,07/18/2017,07/24/2017,complete,Nob Hill,2000.0
4,201611283529,demolitions,11/28/2016,12/01/2017,NaN,issued,Tenderloin,100000.0


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 198900 entries, 0 to 198899
Data columns (total 8 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Permit Number                        198900 non-null  str    
 1   Permit Type Definition               198900 non-null  str    
 2   Filed Date                           198900 non-null  str    
 3   Issued Date                          183960 non-null  str    
 4   Completed Date                       97191 non-null   str    
 5   Current Status                       198900 non-null  str    
 6   Neighborhoods - Analysis Boundaries  197175 non-null  str    
 7   Estimated Cost                       160834 non-null  float64
dtypes: float64(1), str(7)
memory usage: 27.3 MB


In [10]:
df.shape

(198900, 8)

In [11]:
df.columns

Index(['Permit Number', 'Permit Type Definition', 'Filed Date', 'Issued Date',
       'Completed Date', 'Current Status',
       'Neighborhoods - Analysis Boundaries', 'Estimated Cost'],
      dtype='str')

In [12]:
df.isnull().sum()

Permit Number                               0
Permit Type Definition                      0
Filed Date                                  0
Issued Date                             14940
Completed Date                         101709
Current Status                              0
Neighborhoods - Analysis Boundaries      1725
Estimated Cost                          38066
dtype: int64

In [13]:
## Observations

# - The dataset has 198,900 permit records, and for this project I have narrowed it down to 8 columns that are directly relevant to processing time.
# - The basic identification and status fields are complete, so we have a good starting point for the analysis.
# - There are 14,940 missing values in `Issued Date` and 101,709 missing values in `Completed Date`.
# - Most records are marked as either `complete` or `issued`, while some applications are still `filed` or have other statuses such as withdrawn, cancelled, or expired.
# - Because the missing dates may be related to the current status of an application, I don't want to replace them with made-up values.
# - The date columns are currently stored as text, so the next step is to convert them into a proper date format before calculating processing time.

In [14]:
df["Current Status"].value_counts()

Current Status
complete       97077
issued         83559
filed          12043
withdrawn       1754
cancelled       1536
expired         1370
approved         733
reinstated       563
suspend          193
revoked           50
plancheck         16
appeal             2
disapproved        2
incomplete         2
Name: count, dtype: int64

In [15]:
# Date Conversion (For our analysis, these columns need to behave like dates, not ordinary text.)

In [16]:
df["Filed Date"] = pd.to_datetime(df["Filed Date"], errors="coerce")
df["Issued Date"] = pd.to_datetime(df["Issued Date"], errors="coerce")
df["Completed Date"] = pd.to_datetime(df["Completed Date"], errors="coerce")

In [17]:
df[["Filed Date", "Issued Date", "Completed Date"]].dtypes

Filed Date        datetime64[us]
Issued Date       datetime64[us]
Completed Date    datetime64[us]
dtype: object

In [18]:
# Checking For Inconsistency

# 1. A permit cannot be issued before it is applied for
df[df["Issued Date"] < df["Filed Date"]]


,Permit Number,Permit Type Definition,Filed Date,Issued Date,Completed Date,Current Status,Neighborhoods - Analysis Boundaries,Estimated Cost


In [19]:
# No such case where a permit is issued before it is applied for

In [20]:
# 2. A permit should not be completed before it was filed.
df[df["Completed Date"] < df["Filed Date"]]

,Permit Number,Permit Type Definition,Filed Date,Issued Date,Completed Date,Current Status,Neighborhoods - Analysis Boundaries,Estimated Cost
45604,201404223790,otc alterations permit,2014-04-22,2014-04-22,2014-04-18,issued,Russian Hill,1.0
114177,201602119450,additions alterations or repairs,2016-07-19,NaT,2016-07-07,approved,Bernal Heights,15000.0


In [21]:
# 3. Let's verify there aren't any other logical date problem
df[df["Completed Date"] < df["Issued Date"]]

,Permit Number,Permit Type Definition,Filed Date,Issued Date,Completed Date,Current Status,Neighborhoods - Analysis Boundaries,Estimated Cost
14261,201305318338,sign - erect,2013-05-31,2016-05-20,2014-04-01,issued,South of Market,5000.0
30069,201311041086,otc alterations permit,2013-11-04,2015-03-13,2014-06-24,cancelled,Potrero Hill,8000.0
30070,201311041089,otc alterations permit,2013-11-04,2015-03-13,2014-06-24,cancelled,Potrero Hill,7000.0
45604,201404223790,otc alterations permit,2014-04-22,2014-04-22,2014-04-18,issued,Russian Hill,1.0
65317,201410239720,otc alterations permit,2014-10-23,2015-06-15,2015-05-26,issued,Sunset/Parkside,50000.0
65318,201410239720,otc alterations permit,2014-10-23,2015-06-15,2015-05-26,issued,Sunset/Parkside,50000.0
70163,201412123677,otc alterations permit,2014-12-12,2015-12-12,2015-02-12,issued,Marina,2200.0
70164,201412123677,otc alterations permit,2014-12-12,2015-12-12,2015-02-12,issued,Marina,2200.0
73708,201501266602,otc alterations permit,2015-01-26,2015-10-30,2015-02-23,suspend,Oceanview/Merced/Ingleside,20000.0
111527,201508255188,otc alterations permit,2015-08-25,2016-08-25,2016-03-03,issued,Western Addition,32000.0


In [22]:
## Checking Date Consistency Conclusions 

# I found a few records where the completion date comes before the filing or
# issue date. Since that does not make sense for the permit process, I will not
# use those completion dates in our processing-time calculations.

# I will keep the records themselves and only treat the incorrect completion
# dates as missing.

In [23]:
invalid_completed = (
    (df["Completed Date"] < df["Filed Date"]) |
    (df["Completed Date"] < df["Issued Date"])
)

df.loc[invalid_completed, "Completed Date"] = pd.NaT

In [24]:
df[df["Completed Date"] < df["Issued Date"]]

,Permit Number,Permit Type Definition,Filed Date,Issued Date,Completed Date,Current Status,Neighborhoods - Analysis Boundaries,Estimated Cost


In [25]:
# Checking for Duplicates

In [26]:
df.duplicated().sum()

np.int64(17258)

In [27]:
df["Permit Number"].duplicated().sum()

np.int64(17405)

In [28]:
df[df["Permit Number"].duplicated(keep=False)].sort_values("Permit Number").head(20)

,Permit Number,Permit Type Definition,Filed Date,Issued Date,Completed Date,Current Status,Neighborhoods - Analysis Boundaries,Estimated Cost
524,201301027072,otc alterations permit,2013-01-02,2013-01-02,2013-04-18,complete,Castro/Upper Market,1.0
525,201301027072,otc alterations permit,2013-01-02,2013-01-02,2013-04-18,complete,Castro/Upper Market,1.0
531,201301027079,otc alterations permit,2013-01-02,2013-01-02,NaT,issued,Haight Ashbury,18000.0
532,201301027079,otc alterations permit,2013-01-02,2013-01-02,NaT,issued,Haight Ashbury,18000.0
534,201301027081,otc alterations permit,2013-01-02,2013-01-02,NaT,issued,West of Twin Peaks,40000.0
539,201301027081,otc alterations permit,2013-01-02,2013-01-02,NaT,issued,West of Twin Peaks,40000.0
555,201301027093,otc alterations permit,2013-01-02,2013-01-02,2013-03-23,complete,Inner Richmond,3500.0
556,201301027093,otc alterations permit,2013-01-02,2013-01-02,2013-03-23,complete,Inner Richmond,3500.0
560,201301027095,otc alterations permit,2013-01-02,2013-01-02,2013-08-09,complete,Castro/Upper Market,26530.0
567,201301027095,otc alterations permit,2013-01-02,2013-01-02,2013-08-09,complete,Castro/Upper Market,26530.0


In [29]:
## Removing Exact Duplicates

# I found a number of records that are completely identical across all the columns we are using.

# Since these rows contain the same permit information repeated more than once,
# keeping the duplicates would give some applications more weight in the
# analysis than others.

# I will remove only the exact duplicate rows and keep the first occurrence.

In [30]:
df = df.drop_duplicates()

In [31]:
df.duplicated().sum()

np.int64(0)

In [32]:
# Handling the missing values

In [33]:
df["Neighborhoods - Analysis Boundaries"].isnull().sum()

np.int64(1654)

In [34]:
## Handling Missing Neighborhoods

# There are 1,654 missing values in the neighborhood column.

# Since the neighborhood is useful for comparing processing times by location,
# I will keep these records and label the missing values as `Unknown` rather
# than removing the rows or guessing the location.

In [35]:
df["Neighborhoods - Analysis Boundaries"] = df["Neighborhoods - Analysis Boundaries"].fillna("Unknown")

In [36]:
df["Neighborhoods - Analysis Boundaries"].isnull().sum()

np.int64(0)

In [37]:
## Handling Missing Estimated Cost

# The `Estimated Cost` column contains some missing values. Since cost is not
# required for the main processing-time analysis, I will leave these values
# missing rather than replacing them with an assumed value.

# This avoids introducing information that was not present in the original data.

In [38]:
df["Estimated Cost"].isnull().sum()

np.int64(35235)

In [39]:
# As estimated cost is not an much essential factor for our analysis so leaving it as it is there

In [40]:
df[df["Issued Date"].isnull()]["Current Status"].value_counts()

Current Status
filed          10854
withdrawn       1468
approved         624
cancelled        304
reinstated        32
complete          18
plancheck         16
suspend            4
appeal             2
disapproved        2
incomplete         1
Name: count, dtype: int64

In [41]:
## Handling Missing Issued Dates

# There are 14,940 applications with a missing `Issued Date`.

# Most of these records are currently marked as `filed`, which suggests that
# many applications may not have reached the issue stage yet. Since the other
# statuses do not give us enough information to safely reconstruct the missing
# dates, I will keep these values as missing.

# This is important because filling the dates with an assumed value would give
# us incorrect processing times.

In [42]:
df[df["Completed Date"].isnull()]["Current Status"].value_counts()

Current Status
issued         76411
filed          10854
withdrawn       1469
cancelled       1400
expired         1219
approved         628
reinstated       478
suspend          155
revoked           40
plancheck         16
appeal             2
disapproved        2
incomplete         1
Name: count, dtype: int64

In [43]:
## Handling Missing Completed Dates

# Many records do not have a `Completed Date`, and most of these applications
# are not currently marked as complete.

# Since there is no reliable way to determine the missing completion dates, I
# will leave them as missing instead of filling them with an assumed value.

In [44]:
df["Days to Issue"] = (
    df["Issued Date"] - df["Filed Date"]
).dt.days

In [45]:
df["Days to Issue"].describe()

count    168317.000000
mean         24.415448
std          87.825630
min           0.000000
25%           0.000000
50%           0.000000
75%           6.000000
max        1740.000000
Name: Days to Issue, dtype: float64

In [46]:
df["Days to Complete"] = (
    df["Completed Date"] - df["Filed Date"]
).dt.days

In [47]:
df["Days to Complete"].describe()

count    88967.000000
mean       183.329886
std        211.139873
min          0.000000
25%         43.000000
50%        111.000000
75%        246.000000
max       1826.000000
Name: Days to Complete, dtype: float64

In [48]:
df[df["Days to Issue"] < 0]

,Permit Number,Permit Type Definition,Filed Date,Issued Date,Completed Date,Current Status,Neighborhoods - Analysis Boundaries,Estimated Cost,Days to Issue,Days to Complete


In [49]:
df[df["Days to Complete"] < 0]

,Permit Number,Permit Type Definition,Filed Date,Issued Date,Completed Date,Current Status,Neighborhoods - Analysis Boundaries,Estimated Cost,Days to Issue,Days to Complete


In [50]:
df.isnull().sum()

Permit Number                              0
Permit Type Definition                     0
Filed Date                                 0
Issued Date                            13325
Completed Date                         92675
Current Status                             0
Neighborhoods - Analysis Boundaries        0
Estimated Cost                         35235
Days to Issue                          13325
Days to Complete                       92675
dtype: int64

In [51]:
df.shape

(181642, 10)

In [52]:
df.groupby("Permit Type Definition")["Days to Issue"].agg(
    ["count", "mean", "median"]
).sort_values("median", ascending=False)

,count,mean,median
Permit Type Definition,,,
new construction,149,449.496644,378.0
new construction wood frame,481,409.133056,303.0
demolitions,336,353.264881,227.0
additions alterations or repairs,7874,247.416434,197.0
grade or quarry or fill or excavate,70,93.014286,64.0
sign - erect,2380,52.396639,10.0
wall or painted sign,358,45.684358,9.0
otc alterations permit,156669,10.412673,0.0


In [53]:
permit_summary = df.groupby("Permit Type Definition").agg(
    Applications=("Permit Number", "count"),
    Avg_Days_to_Issue=("Days to Issue", "mean"),
    Median_Days_to_Issue=("Days to Issue", "median")
).reset_index()

In [54]:
permit_summary.sort_values(
    "Applications",
    ascending=False
).head(10)

,Permit Type Definition,Applications,Avg_Days_to_Issue,Median_Days_to_Issue
5,otc alterations permit,164222,10.412673,0.0
0,additions alterations or repairs,12359,247.416434,197.0
6,sign - erect,2858,52.396639,10.0
4,new construction wood frame,780,409.133056,303.0
1,demolitions,538,353.264881,227.0
7,wall or painted sign,506,45.684358,9.0
3,new construction,297,449.496644,378.0
2,grade or quarry or fill or excavate,82,93.014286,64.0


In [55]:
permit_summary.sort_values(
    "Median_Days_to_Issue",
    ascending=False
).head(10)

,Permit Type Definition,Applications,Avg_Days_to_Issue,Median_Days_to_Issue
3,new construction,297,449.496644,378.0
4,new construction wood frame,780,409.133056,303.0
1,demolitions,538,353.264881,227.0
0,additions alterations or repairs,12359,247.416434,197.0
2,grade or quarry or fill or excavate,82,93.014286,64.0
6,sign - erect,2858,52.396639,10.0
7,wall or painted sign,506,45.684358,9.0
5,otc alterations permit,164222,10.412673,0.0


In [56]:
df["Year"] = df["Filed Date"].dt.year

In [57]:
df.groupby("Year")["Days to Issue"].median()

Year
2013    0.0
2014    0.0
2015    0.0
2016    0.0
2017    0.0
2018    0.0
Name: Days to Issue, dtype: float64

In [58]:
df.groupby("Neighborhoods - Analysis Boundaries")["Days to Issue"].agg(
    ["count", "mean", "median"]
).sort_values("median", ascending=False).head(10)

,count,mean,median
Neighborhoods - Analysis Boundaries,,,
Lakeshore,1012,36.632411,5.0
Mission Bay,2010,30.842786,2.0
Golden Gate Park,57,27.807018,1.0
Financial District/South Beach,20074,20.079755,1.0
Bernal Heights,5093,27.102297,0.0
Bayview Hunters Point,4210,39.890736,0.0
Excelsior,2818,21.169624,0.0
Glen Park,2276,29.006591,0.0
Chinatown,3213,17.786492,0.0


In [59]:
## Moving from Python to SQL

# After cleaning the data and creating the main processing-time metrics in
# Python, I am moving the prepared dataset into SQL.

# The goal here is to answer the main business questions using SQL rather than
# doing every calculation in Pandas.

In [60]:
# 9. Rename columns to SQL-friendly names
df.columns = [
    "permit_number",
    "permit_type",
    "filed_date",
    "issued_date",
    "completed_date",
    "current_status",
    "neighborhood",
    "estimated_cost",
    "days_to_issue",
    "days_to_complete",
    "year"
]

In [62]:
# Also create a semicolon-separated copy for MySQL import
df.to_csv(
    "cleaned_building_permits_mysql.csv",
    index=False,
    sep=";",
 na_rep="NULL")